# Working with real data

Real data frames are wide, hold mixed types, contain missing values, and
carry structure that a purely additive model cannot express. This chapter
covers the features that address each of those: whole-frame formulas,
inline transforms, categorical predictors, a missing-value policy,
interaction learners, distribution-free prediction intervals, warm
continuation of the boosting path, and a portable model artifact.


In [1]:
import numpy as np
import polars as pl
import mboostrs as mb

rng = np.random.default_rng(0)
n = 600
age = rng.uniform(20, 70, n)
income = rng.uniform(1.0, 10.0, n)          # positive; used with log()
region = np.array(["north", "south", "east", "west"])[rng.integers(0, 4, n)]
noise = rng.normal(scale=0.4, size=n)
y = 0.03 * age + 1.2 * np.log(income) + np.where(region == "south", 0.8, -0.2) + noise
data = pl.DataFrame({"age": age, "income": income, "region": region, "y": y})
data.head()


age,income,region,y
f64,f64,str,f64
51.848084,1.055591,"""south""",2.538933
33.489336,7.490492,"""south""",4.307664
22.048676,7.08944,"""north""",2.725373
20.826382,6.912113,"""south""",3.504169
60.663512,7.186735,"""north""",3.880269


## Whole-frame formulas

`y ~ .` expands to every column except the response, in column order.
`. - z` drops a column; `. + bbs(x, df=6)` gives one predictor a specific
learner while the rest keep the default. A bare column name becomes a
P-spline term in `gamboost` and a linear term in `glmboost`.


In [2]:
# every numeric column as a smooth term, minus one, plus an explicit spline
num = data.select(["age", "income", "y"])
fit = mb.gamboost("y ~ . - income + bbs(income, df=6)", data=num,
                  family=mb.Gaussian(), control=mb.boost_control(200, 0.1))
fit.base_learner_names()


['age', 'income']

## Inline transforms

A formula may name a transformed predictor directly, and the transform is
materialized into a derived column before fitting:

```python
mb.glmboost("y ~ log(income) + scale(age)", data=data, family=mb.Gaussian())
```

### Supported transforms

| Written in the formula | Builds |
|---|---|
| `log(x)`, `log1p(x)`, `sqrt(x)`, `exp(x)` | the elementwise function |
| `scale(x)` | `(x - mean) / sd`, from the **training** mean and sd |
| `center(x)` | `x - mean`, from the training mean |
| `pmin(x, c)`, `pmax(x, c)` | `x` clamped below or above at the constant `c` |
| `ecdf(x)` | the share of **training** rows at or below `x`, between 0 and 1 |
| `I(<expression>)` | arithmetic over data columns: `I(x**2)`, `I(x * z)`, `I(x * 2 + 1)` |

`I()` accepts `+ - * / % ** ()` and numeric literals over any number of data
columns. It also accepts R's `^` for exponentiation, so `I(x^2)` and `I(x**2)`
are the same term.

### Stored transform parameters

`scale(x)` divides by the standard deviation **computed on the training data**,
and that number is recorded on the fitted model. When `predict(newdata=...)`
runs, the recorded value is reused rather than recomputed. Recomputing it would
map the same covariate value to a different number depending on which rows
happened to share the prediction batch — the error that pre-transforming a
column by hand invites, and the reason to write the transform in the formula
instead of preparing the column yourself.

Terms are reported as you wrote them. `log(x)` becomes a column named `log__x`
internally, because the rewritten formula has to stay parseable, but
`coef()`, `selected`, and plot labels all say `log(x)`, which is what R shows.

(inline-transforms-identity)=
### Arithmetic expressions: `I()`

R evaluates a formula with `model.matrix`, so R's `I()` runs arbitrary R code:
`I(ifelse(x > 0, log(x), 0))` works there. Here `I()` is restricted to
arithmetic over data columns.

The restriction has two effects. A formula stays safe to accept from a config
file or an HTTP request, because it cannot call a function. And every term
stays replayable: the model records the expression, so a saved model rebuilds
the same column months later without needing the code that produced it.

As a consequence, a transform involving a function call, a conditional, or a
lookup has to be computed as a column before fitting. That column is then an
ordinary predictor, and you are responsible for computing it the same way at
prediction time. Inline transforms preserve the same transform definition for
fitting and prediction; use the formula when it can express the transform.

```{note}
Two R formula constructions have no equivalent here. `x:z` and `x*z` as
*formula operators* — R's interaction notation — are not parsed; use
`I(x*z)` for the product column, or `%X%` for a tensor-product interaction
between two base learners, covered under [](#interaction-learners) below.
And `poly(x, 3)`, R's orthogonal polynomial basis,
is not available; `bbs(x)` is the better tool for a smooth term, and
`I(x**2) + I(x**3)` gives the raw polynomial.
```


In [3]:
fit_log = mb.glmboost("y ~ log(income) + scale(age)", data=data,
                      family=mb.Gaussian(), control=mb.boost_control(300, 0.1))

# predict two rows alone, then the same two rows inside a wider batch
small = {"income": np.array([2.0, 8.0]), "age": np.array([30.0, 60.0])}
wide = {"income": np.array([2.0, 8.0, 100.0]), "age": np.array([30.0, 60.0, 200.0])}
p_small = np.asarray(fit_log.predict(newdata=small, type="response"))
p_wide = np.asarray(fit_log.predict(newdata=wide, type="response"))[:2]
float(np.max(np.abs(p_small - p_wide)))   # 0.0: batch-independent


0.0

### Heavy-tailed predictors: `ecdf(x)`

`bbs(x)` places its knots evenly over the range of `x`. When a few rows sit
far from the rest, as with a household-occupancy column whose bulk lies below
5 and whose maximum is 600, most knot intervals hold no data, and a
cross-validation fold whose training rows leave a gap extends the spline
across it; the held-out predictions for the extreme rows can then be large
enough to decide `mstop` for the whole model.

`ecdf(x)` maps each value to the share of training rows at or below it. The
model records up to 257 training quantiles, interpolates linearly between
them, and holds the value at 0 below the training minimum and at 1 above the
maximum. `bbs(ecdf(x))` therefore has its knots at quantiles of `x`, so each
knot interval holds about the same number of rows, and a new value beyond
the training range receives the effect at the nearest training extreme, as a
tree or an explainable boosting machine (EBM) gives it. The effect is estimated as a function of the quantile,
so its plot has `ecdf(x)` on the horizontal axis; `bbs(log(x))` keeps a
measurement scale and is the first choice for a positive right-skewed
column. On California housing, `bbs(ecdf(x))` on every column reached a
held-out R² of 0.73 on three splits, where `bbs(x)` reached between −0.19
and 0.66, depending on which rows each fold held out.

In [4]:
skewed_rng = np.random.default_rng(11)
occupancy = skewed_rng.lognormal(mean=1.0, sigma=1.2, size=n)
occ_data = {"occupancy": occupancy,
            "y": np.log(occupancy) + skewed_rng.normal(scale=0.3, size=n)}
fit_ecdf = mb.gamboost("y ~ bbs(ecdf(occupancy), df=4)", data=occ_data,
                       control=mb.boost_control(300, 0.1))

# a value far above the training maximum gets the effect at the maximum
beyond = {"occupancy": np.array([occupancy.max(), 50 * occupancy.max()])}
np.round(fit_ecdf.predict(newdata=beyond), 4)

array([3.4971, 3.4971])

## Categorical predictors

A string or categorical column is encoded for the learners that treat its
levels as unordered. In `gamboost`, `bols(region)` with no other argument,
a bare `region` term, and `y ~ .` fit one effect per level: R's `bols` on a
factor, whose fitted path mboostrs reproduces (the coefficients are reported
per level rather than as contrasts against a reference level).

`bols(region, intercept=False)`, `bols(region, df=2)` and
`bols(region, lambda=10)` use mboost's treatment-contrast design instead:
the columns `[1, d_2, ..., d_K]` (without the leading 1 when
`intercept=False`, so the first level is the zero row), with a ridge
penalty on every column when `df` or `lambda` is given.

`brandom(region)` fits the level effects with a ridge penalty that shrinks
them towards each other, and `bmrf` shrinks neighbouring levels together. In
`glmboost`, a factor term expands into R's treatment-contrast columns
(`regionnorth`, `regionsouth`, ...), each its own component.

A spline or
`bmono` rejects a categorical column, because
encoding its levels as numbers would impose an order the data does not
have. An unseen level at prediction time raises an error naming the level.
For a `brandom` column, `predict(..., unseen_levels="average")` predicts a
row with an unseen level as a member of an average group, the mean of the
term's level coefficients, which is the prediction for a group the fit never
saw.

In [5]:
fit_region = mb.gamboost("y ~ bbs(age, df=4) + brandom(region)", data=data,
                         family=mb.Gaussian(), control=mb.boost_control(200, 0.1))
coefs = fit_region.coef()
{k: (np.round(v, 3).tolist() if np.ndim(v) else round(float(v), 3))
 for k, v in coefs.items()}


{'age': [-0.373,
  -0.426,
  -0.479,
  -0.529,
  -0.559,
  -0.551,
  -0.494,
  -0.386,
  -0.263,
  -0.173,
  -0.116,
  -0.074,
  -0.024,
  0.058,
  0.153,
  0.234,
  0.305,
  0.366,
  0.411,
  0.447,
  0.469,
  0.486,
  0.513,
  0.541],
 'brandom(region)': [-0.131, -0.147, 0.702, -0.308],
 '(offset)': 3.249}

## Missing values

By default a non-finite value in a model variable raises an error naming
the column, the count, and the first offending row. `na_action="omit"`
instead drops the rows with a missing value in a referenced column and
records the count in `summary()`. `na_action="impute"` keeps the rows: each
numeric predictor with a missing value is filled with its training median,
and an indicator `<x>_missing` (1 where the value was missing) enters the
model as its own term. A categorical predictor's missing values become a
level of their own, `"(missing)"`, which is never the reference level of a
treatment contrast. These steps are replayed on new data, so the model
predicts rows with missing values, which `"omit"` cannot. A missing value in
a column the formula does not use is ignored, and a missing response always
drops its row.

In [6]:
holed = data.clone().to_dict(as_series=False)
holed["age"] = np.array(holed["age"]); holed["age"][3] = np.nan
holed["income"] = np.array(holed["income"]); holed["income"][17] = np.nan

fit_omit = mb.glmboost("y ~ log(income) + age", data=holed, family=mb.Gaussian(),
                       control=mb.boost_control(100, 0.1), na_action="omit")
info = fit_omit.summary(as_dict=True)
(info["n"], info["n_omitted"])


(598, 2)

With `"impute"` the same data keep every row, and the indicator's
coefficient says how rows with a missing income differ from rows at the
median income:

In [7]:
fit_impute = mb.glmboost("y ~ income + age", data=holed, family=mb.Gaussian(),
                         control=mb.boost_control(100, 0.1), na_action="impute")
{name: round(float(value), 4) for name, value in fit_impute.coef().items()}

{'(Intercept)': -2.6462,
 'income': 0.2473,
 'age': 0.0279,
 'income_missing': -0.7241}

A missing region becomes the level `(missing)`, and its contrast
coefficient `region_missing` compares those rows with the reference level:

In [8]:
holed["region"] = np.array(holed["region"], dtype=object)
holed["region"][[5, 40, 77]] = None
fit_levels = mb.glmboost("y ~ income + age + region", data=holed, family=mb.Gaussian(),
                         control=mb.boost_control(100, 0.1), na_action="impute")
sorted(fit_levels.coef())

['(Intercept)',
 'age',
 'age_missing',
 'income',
 'income_missing',
 'region_missing',
 'regionsouth',
 'regionwest']

(interaction-learners)=
## Interaction learners

The `%X%` operator fits the interaction of two base learners: the row-wise
tensor of their designs with a Kronecker-sum penalty. `bbs(x) %X%
brandom(g)` fits a smooth curve whose shape varies by group; `bbs(x) %X%
bbs(z)` fits an interaction surface with independent marginal penalties.
The interaction captures structure that the corresponding additive model
cannot.

The tensor has one column per pair of marginal columns. A default `bbs` has
20 inner knots and 24 columns, so `bbs(x) %X% bbs(z)` has 24 × 24 = 576
columns, and every boosting step solves a 576 × 576 system. Five inner knots
per margin, `bbs(x, df=2.5, knots=5) %X% bbs(z, df=2.5, knots=5)`, give an
9 × 9 = 81-column surface; on the interaction task of
[](../cases/benchmark_insight.ipynb) that fit took 0.22 s against 5.4 s for
the default margins, with a slightly lower excess error. The terms
`autospec(interactions="adopt")` adds are written this way.


In [9]:
# a slope that differs by region: additive terms cannot represent it
slope = {"north": 1.0, "south": -1.0, "east": 2.0, "west": 0.0}
yx = np.array([slope[r] for r in region]) * (age - age.mean()) / 10 + noise
idata = {"age": age, "region": region, "y": yx}

inter = mb.gamboost("y ~ bbs(age, df=4) %X% brandom(region)", data=idata,
                    family=mb.Gaussian(), control=mb.boost_control(400, 0.1))
add = mb.gamboost("y ~ bbs(age, df=4) + brandom(region)", data=idata,
                  family=mb.Gaussian(), control=mb.boost_control(400, 0.1))

def r2(m):
    return 1 - np.sum((yx - m.fitted_) ** 2) / np.sum((yx - yx.mean()) ** 2)

{"interaction": round(r2(inter), 3), "additive": round(r2(add), 3)}


{'interaction': np.float64(0.95), 'additive': np.float64(0.172)}

## Prediction intervals

A standard model fits the conditional mean; `confint` describes uncertainty
about the fitted effects, not where a new observation will fall.
`fit.conformal_interval` returns a distribution-free predictive interval (an interval whose width is set from the errors on held-out rows, with no assumed error distribution) with
finite-sample marginal coverage (the stated share of new observations falls inside on average over the input values, at any sample size), with no assumption that the early-stopping
iteration was chosen well.

Three methods are available: `split` (one extra fit, constant width),
`cv+` (uses every observation for calibration), and `cqr` (conformalized
quantile regression, width that adapts to the local noise).


In [10]:
fit_mean = mb.gamboost("y ~ bbs(age, df=4) + bbs(income, df=4)", data=data,
                       family=mb.Gaussian(), control=mb.boost_control(200, 0.1))
ci = fit_mean.conformal_interval(level=0.9, method="cv+", seed=1, n_folds=5)
coverage = ci.empirical_coverage(data["y"].to_numpy())
{"target": 0.9, "empirical_coverage": round(coverage, 3),
 "mean_width": round(float(np.mean(ci.width)), 3)}


{'target': 0.9, 'empirical_coverage': 0.913, 'mean_width': 1.973}

## Extending the boosting path

`fit` fixes `mstop` in advance. When the risk path shows the optimum at the
boundary, `continue_to(N)` grows the model to a larger `mstop` without
re-specifying it. The result equals a cold fit with `mstop=N`.


In [11]:
short = mb.gamboost("y ~ bbs(age, df=4) + bbs(income, df=4)", data=data,
                    family=mb.Gaussian(), control=mb.boost_control(100, 0.1))
longer = short.continue_to(400)
cold = mb.gamboost("y ~ bbs(age, df=4) + bbs(income, df=4)", data=data,
                   family=mb.Gaussian(), control=mb.boost_control(400, 0.1))
{"grown_mstop": longer.mstop,
 "matches_cold_fit": bool(np.max(np.abs(longer.fitted_ - cold.fitted_)) == 0.0)}


{'grown_mstop': 400, 'matches_cold_fit': True}

## Component subsampling: `subsample=`

On a model with many candidate learners, `subsample=f` evaluates a random
fraction `f` of the candidates for selection at each iteration. This
reduces per-iteration cost and perturbs selection frequencies. It is off by
default; `subsample_seed` makes the path reproducible. The count of
subsampled iterations is returned by the fitted model's `execution_report()` method, as `component_subsample_iterations`.


In [12]:
wide_rng = np.random.default_rng(3)
p = 40
cols = {f"x{j}": wide_rng.normal(size=n) for j in range(p)}
cols["y"] = 1.5 * cols["x0"] - 0.9 * cols["x1"] + 0.6 * cols["x2"] \
    + wide_rng.normal(scale=0.3, size=n)
rhs = " + ".join(f"x{j}" for j in range(p))
sub = mb.glmboost(f"y ~ {rhs}", data=cols, family=mb.Gaussian(),
                  control=mb.boost_control(300, 0.1), subsample=0.3, subsample_seed=42)
top3 = sorted(sub.coefficients_, key=lambda k: -abs(sub.coefficients_[k]))[:3]
{"top_3_by_coef": sorted(top3),
 "subsampled_iterations": sub.execution_report().component_subsample_iterations}


{'top_3_by_coef': ['x0', 'x1', 'x2'], 'subsampled_iterations': 300}

## Portable model artifacts

`fit.save` writes a schema-versioned JSON artifact holding the formula,
family, learner specifications, coefficient path, transform recipe, and
execution report. `load_model` reconstructs a model that predicts
identically. Unlike a pickle, the artifact does not depend on the Python
version or the compiled extension, and its contents are inspectable.


In [13]:
import tempfile, os
path = os.path.join(tempfile.mkdtemp(), "model.json")
fit_region.save(path)
restored = mb.load_model(path)

grid = {"age": np.array([25.0, 55.0]), "region": np.array(["north", "south"])}
same = np.array_equal(
    np.asarray(fit_region.predict(newdata=grid, type="response")),
    np.asarray(restored.predict(newdata=grid, type="response")),
)
{"round_trip_identical": bool(same), "artifact_kib": round(os.path.getsize(path) / 1024, 1)}


{'round_trip_identical': True, 'artifact_kib': 156.7}